
# Assignment 10: MT19937 Stream Cipher and Predictable Token Attack

## 1. Introduction

This assignment demonstrates the security weaknesses of using a statistical pseudo-random number generator (PRNG) in cryptographic applications.

We use the MT19937 algorithm to:

1. Generate a keystream for a toy stream cipher.
2. Encrypt and decrypt plaintext using XOR.
3. Recover a 16-bit seed using a brute-force attack and known plaintext.
4. Generate a predictable token using a Unix timestamp as the seed.
5. Recover the timestamp by testing possible seed values.

The experiments demonstrate why a statistical PRNG should not be used for cryptographic security.



## 2. What is a Pseudo-Random Number Generator?

A Pseudo-Random Number Generator (PRNG) is an algorithm that generates a sequence of numbers that appears random.

However, its output is deterministic because it is initialized using an initial value called a seed.

### Important characteristics

- A PRNG uses an initial seed.
- The same seed produces the same sequence.
- Its output is deterministic.
- Its output may appear statistically random but is not necessarily secure against prediction.

### Example

In Python:

```python
import random

rng1 = random.Random(1234)
rng2 = random.Random(1234)

print(rng1.getrandbits(8))
print(rng2.getrandbits(8))
```

Both generators produce the same first output because they use the same seed.



## 3. MT19937 Algorithm

MT19937 is a widely used pseudo-random number generator belonging to the Mersenne Twister family.

Important characteristics:

- It has a state size of 19937 bits.
- It has a period of 2^19937 - 1.
- It produces deterministic pseudo-random sequences.
- Python's random module uses the Mersenne Twister algorithm by default.

### Why is MT19937 unsuitable for cryptography?

MT19937 is designed for statistical randomness, not cryptographic security.

Its deterministic nature and predictable state make it unsuitable for generating security-sensitive values such as authentication tokens, session identifiers and cryptographic keys.

For cryptographic applications, a Cryptographically Secure Pseudo-Random Number Generator (CSPRNG) should be used.



## 4. XOR-Based Stream Cipher

A stream cipher encrypts plaintext by combining each byte of the plaintext with a corresponding byte from a keystream.

In this assignment, MT19937 generates the keystream, and XOR is used for encryption and decryption.

### 4.1 XOR Operation

XOR (Exclusive OR) is a bitwise operation.

Its truth table is:

| A | B | A XOR B |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

### 4.2 Encryption Formula

Let:

- P = Plaintext byte
- K = Keystream byte
- C = Ciphertext byte

Encryption is performed using:

C = P XOR K

### 4.3 Decryption Formula

Decryption uses the same XOR operation:

P = C XOR K

This works because XOR has the following property:

(P XOR K) XOR K = P

Therefore, applying XOR twice with the same keystream recovers the original plaintext.

### 4.4 Example

Suppose:

Plaintext byte = 72 (ASCII character H)

Keystream byte = 247

In binary:

Plaintext = 01001000

Keystream = 11110111

XOR result:

Ciphertext = 10111111

In decimal:

72 XOR 247 = 191

For decryption:

191 XOR 247 = 72

The original character H is recovered.

### 4.5 Keystream Generation

In Python, MT19937 is initialized using:

```python
rng = random.Random(1234)
```

One 8-bit keystream value is generated using:

```python
key_byte = rng.getrandbits(8)
```

We generate one keystream byte for every plaintext byte.

The same seed and byte-generation procedure must be used for encryption and decryption.



## 5. Known Plaintext Attack

A Known Plaintext Attack (KPA) is a cryptanalytic attack in which an attacker knows some part of the original plaintext and its corresponding ciphertext.

In our experiment, the plaintext contains 14 consecutive A characters:

AAAAAAAAAAAAAA

The attacker knows this pattern exists but does not know its position in the plaintext.

### 5.1 Recovering the Keystream

The encryption formula is:

C = P XOR K

Therefore:

K = C XOR P

If the attacker knows a plaintext byte and its corresponding ciphertext byte, the keystream byte can be calculated.

### 5.2 Why We Use 14 Consecutive A Characters

The assignment uses 14 consecutive A characters as a known plaintext marker.

The attacker decrypts the ciphertext using different candidate seeds and searches for this marker.

A matching marker, together with the expected plaintext structure, helps identify the correct candidate seed.

### 5.3 Important Observation

Knowing the plaintext pattern does not automatically reveal the complete MT19937 seed.

In our experiment, we combine the known plaintext pattern with brute-force searching over the 16-bit seed space.



## 6. Brute Force Seed Recovery

In Part A, the stream cipher uses a randomly selected 16-bit seed.

A 16-bit seed has the following possible values:

0 to 65535

Total possible seeds:

2^16 = 65,536

### 6.1 Attack Methodology

The attacker performs the following operations:

1. Obtain the ciphertext.
2. Use the known plaintext pattern of 14 consecutive A characters.
3. Iterate through all possible 16-bit seeds.
4. Initialize MT19937 using each candidate seed.
5. Generate the keystream using getrandbits(8).
6. Decrypt the ciphertext using XOR.
7. Search the candidate plaintext for the known pattern.
8. Validate the expected plaintext structure.
9. Report the matching seed and recovered plaintext.

### 6.2 Why Brute Force Works Here

The seed space is limited to 65,536 possibilities.

Therefore, every possible seed can be tested.

Once the correct seed is found, the deterministic MT19937 generator reproduces the same keystream, allowing the complete plaintext to be recovered.


### 6.3 Practical Implementation: Part A

The following cells run the stream cipher and the exhaustive 16-bit seed-recovery attack. The recovery function records a match but continues testing all 65,536 candidates.

In [9]:
import random
import secrets
import string

def generate_keystream(length, seed):
    rng = random.Random(seed)
    return [rng.getrandbits(8) for _ in range(length)]

def encrypt_decrypt(data, seed):
    rng = random.Random(seed)
    result = bytearray()

    for byte in data:
        key_byte = rng.getrandbits(8)
        result.append(byte ^ key_byte)

    return bytes(result)

def generate_special_plaintext():
    characters = string.ascii_letters + string.digits + string.punctuation + " "
    prefix_length = secrets.randbelow(16) + 5
    prefix = "".join(secrets.choice(characters) for _ in range(prefix_length))
    return (prefix + "A" * 14).encode("ascii")


plaintext = generate_special_plaintext()
seed = secrets.randbelow(2**16)

ciphertext = encrypt_decrypt(plaintext, seed)
decrypted = encrypt_decrypt(ciphertext, seed)

print("PART A: MT19937 STREAM CIPHER")
print("Original Plaintext:", plaintext.decode("ascii"))
print("Plaintext Length:", len(plaintext))
print("Random 16-bit Seed:", seed)
print("Ciphertext (Hex):", ciphertext.hex())
print("Decrypted Plaintext:", decrypted.decode("ascii"))
print("Encryption/Decryption Verification:", decrypted == plaintext)
print("Known Pattern Found:", b"A" * 14 in plaintext)


PART A: MT19937 STREAM CIPHER
Original Plaintext: ;[g>m{pPdz!{"AAAAAAAAAAAAAA
Plaintext Length: 27
Random 16-bit Seed: 49707
Ciphertext (Hex): ba99e9416dabe78cef40cfe927ece22062f030406cccae632d6b76
Decrypted Plaintext: ;[g>m{pPdz!{"AAAAAAAAAAAAAA
Encryption/Decryption Verification: True
Known Pattern Found: True


In [10]:
def recover_seed(ciphertext):
    known_pattern = b"A" * 14
    tested_seeds = 0
    recovered_seed = None
    recovered_plaintext = None


    for candidate_seed in range(2**16):
        candidate_plaintext = encrypt_decrypt(ciphertext, candidate_seed)
        tested_seeds += 1

        position = candidate_plaintext.find(known_pattern)

        if position != -1:
            prefix = candidate_plaintext[:position]
            valid_length = 5 <= len(prefix) <= 20
            printable = all(32 <= byte <= 126 for byte in prefix)
            valid_structure = (
                valid_length
                and printable
                and position + 14 == len(candidate_plaintext)
            )

        
            if valid_structure and recovered_seed is None:
                recovered_seed = candidate_seed
                recovered_plaintext = candidate_plaintext

    return recovered_seed, recovered_plaintext, tested_seeds

recovered_seed, recovered_plaintext, tested = recover_seed(ciphertext)

print("PART A: BRUTE-FORCE SEED RECOVERY")
print("Total Seeds Tested:", tested)
print("Recovered Seed:", recovered_seed)
print(
    "Recovered Plaintext:",
    recovered_plaintext.decode("ascii") if recovered_plaintext else None
)
print("Known Pattern Found:", recovered_plaintext is not None and b"A" * 14 in recovered_plaintext)

if recovered_seed is not None:
    verification = encrypt_decrypt(ciphertext, recovered_seed)
    print("Full Plaintext Verification:", verification == plaintext)
    print("Recovered Seed Matches Original:", recovered_seed == seed)


PART A: BRUTE-FORCE SEED RECOVERY
Total Seeds Tested: 65536
Recovered Seed: 49707
Recovered Plaintext: ;[g>m{pPdz!{"AAAAAAAAAAAAAA
Known Pattern Found: True
Full Plaintext Verification: True
Recovered Seed Matches Original: True



## 7. Unix Timestamp

A Unix timestamp represents the number of seconds elapsed since 1 January 1970, 00:00:00 UTC.

Python provides the current Unix timestamp using the time module.

```python
import time

timestamp = int(time.time())
print(timestamp)
```

In Part B, this timestamp is used as the seed for MT19937.

Therefore, the random sequence generated by MT19937 depends on the time at which the token is generated.



## 8. Predictable Token Generation

In this experiment, MT19937 is initialized using the current Unix timestamp.

The generator produces 16 consecutive 8-bit values.

Each generated value represents one byte.

Therefore, the token contains:

16 × 8 = 128 bits

The 16 bytes are converted into a hexadecimal representation.

Each byte requires two hexadecimal characters, so the final token contains 32 hexadecimal characters.

### Token Generation Procedure

1. Record the current Unix timestamp T.
2. Initialize MT19937 using T as the seed.
3. Generate 16 consecutive 8-bit outputs using getrandbits(8).
4. Combine the outputs into a 16-byte token.
5. Convert the token into hexadecimal.
6. Display the timestamp and token.



## 9. Timestamp-Based Prediction Attack

The attacker knows that:

- MT19937 was used to generate the token.
- The Unix timestamp was used as the seed.
- The token contains 16 consecutive 8-bit outputs.
- The token was generated within a known time interval.

For this experiment, the attacker searches timestamps from T - 300 seconds to T + 300 seconds.

### 9.1 Search Space

Start timestamp = T - 300

End timestamp = T + 300

Total candidate timestamps:

300 + 300 + 1 = 601

### 9.2 Attack Methodology

1. Obtain the target token in hexadecimal.
2. Obtain an approximate generation timestamp T.
3. Calculate the search interval from T - 300 to T + 300.
4. Initialize MT19937 with each candidate timestamp.
5. Generate 16 consecutive 8-bit outputs.
6. Convert the generated bytes into hexadecimal.
7. Compare the generated token with the target token.
8. If they match, report the candidate timestamp.
9. Regenerate the token using the recovered timestamp to verify the result.

### 9.3 Why Does the Attack Work?

MT19937 is deterministic.

When the same timestamp is used as the seed, the generator produces the same sequence of bytes.

If the attacker can narrow down the possible timestamp values, they can test each candidate and identify the matching token.

### 9.4 Security Implication

Using a timestamp as a seed makes the possible seed values predictable.

This approach is unsuitable for security-sensitive tokens because an attacker may estimate the generation time and reproduce the token.


### 9.5 Practical Implementation: Part B

The following cells generate a timestamp-seeded token and test the complete inclusive interval of 601 timestamp candidates.

In [11]:
import random
import time

def generate_token_from_timestamp(timestamp):
    rng = random.Random(timestamp)
    token_bytes = bytearray()

    for _ in range(16):
        token_bytes.append(rng.getrandbits(8))

    return bytes(token_bytes).hex()


token_timestamp = int(time.time())
target_token = generate_token_from_timestamp(token_timestamp)

print("PART B: PREDICTABLE TOKEN GENERATION")
print("Unix Timestamp:", token_timestamp)
print("Generated Token (Hex):", target_token)
print("Token Length:", len(bytes.fromhex(target_token)), "bytes")


PART B: PREDICTABLE TOKEN GENERATION
Unix Timestamp: 1795009562
Generated Token (Hex): 7405d4cc7ce847d4d45ead2089de48dc
Token Length: 16 bytes


In [12]:
def predict_token(target_token, approximate_timestamp):
    start_time = approximate_timestamp - 300
    end_time = approximate_timestamp + 300
    tested_timestamps = 0
    recovered_timestamp = None


    for candidate_timestamp in range(start_time, end_time + 1):
        generated_token = generate_token_from_timestamp(candidate_timestamp)
        tested_timestamps += 1

     
        if generated_token == target_token and recovered_timestamp is None:
            recovered_timestamp = candidate_timestamp

    return recovered_timestamp, tested_timestamps

recovered_timestamp, tested = predict_token(target_token, token_timestamp)

print("PART B: TIMESTAMP PREDICTION ATTACK")
print("Search Start:", token_timestamp - 300)
print("Search End:", token_timestamp + 300)
print("Total Timestamps Tested:", tested)
print("Recovered Timestamp:", recovered_timestamp)

if recovered_timestamp is not None:
    regenerated_token = generate_token_from_timestamp(recovered_timestamp)
    print("Regenerated Token:", regenerated_token)
    print("Token Verification:", regenerated_token == target_token)
else:
    print("No matching timestamp found.")


PART B: TIMESTAMP PREDICTION ATTACK
Search Start: 1795009262
Search End: 1795009862
Total Timestamps Tested: 601
Recovered Timestamp: 1795009562
Regenerated Token: 7405d4cc7ce847d4d45ead2089de48dc
Token Verification: True


## 10. Pseudocode

### 10.1 Part A: Brute Force Seed Recovery

```text
INPUT: Ciphertext
KNOWN_PATTERN = 14 consecutive A characters

recovered_seed = NONE
recovered_plaintext = NONE

FOR seed FROM 0 TO 65535:

    Initialize MT19937(seed)
    Generate one 8-bit keystream value per ciphertext byte
    Decrypt ciphertext using XOR
    Search for KNOWN_PATTERN in plaintext

    IF pattern is found AND plaintext structure is valid:
        IF recovered_seed is NONE:
            Store seed and plaintext

    CONTINUE testing all remaining seeds

OUTPUT total seeds tested = 65536

IF recovered_seed is not NONE:
    OUTPUT recovered seed and plaintext
    Verify decryption again using recovered seed
ELSE:
    OUTPUT "No matching seed found"
```

### 10.2 Part B: Predictable Token Attack

```text
INPUT: Approximate timestamp T
INPUT: Target token

START = T - 300
END = T + 300
recovered_timestamp = NONE

FOR timestamp FROM START TO END, inclusive:

    Initialize MT19937(timestamp)
    Generate 16 consecutive 8-bit outputs
    Combine outputs into a 16-byte token
    Convert token into hexadecimal

    IF generated token equals target token:
        IF recovered_timestamp is NONE:
            Store timestamp

    CONTINUE testing all remaining timestamps

OUTPUT total timestamps tested = 601

IF recovered_timestamp is not NONE:
    Regenerate token using recovered timestamp
    OUTPUT recovered timestamp and verification result
ELSE:
    OUTPUT "No matching timestamp found"
```


## 11. Security Analysis

The experiments demonstrate two weaknesses of using MT19937 in cryptographic applications.

### Part A: Weak Seed Space

- The seed is selected from a 16-bit range.
- Only 65,536 possible seeds exist.
- An attacker can test all possible seeds.
- Known plaintext helps identify the correct candidate.

### Part B: Predictable Timestamp Seed

- The timestamp is used as the seed.
- The attacker can estimate the token generation time.
- Only 601 candidate timestamps need to be tested in the specified interval.
- The deterministic output allows the attacker to reproduce the token.

### Recommended Secure Alternative

For security-sensitive random values, use a Cryptographically Secure Pseudo-Random Number Generator (CSPRNG).

In Python, the secrets module provides cryptographically secure random values.

For example:

```python
import secrets

secure_token = secrets.token_hex(16)
print(secure_token)
```

This generates a 16-byte cryptographically secure token represented by 32 hexadecimal characters.

## 12. Conclusion

In this assignment, we implemented an MT19937-based toy stream cipher and demonstrated two attacks.

In Part A, we recovered a 16-bit seed through brute-force searching using a known plaintext pattern.

In Part B, we demonstrated how a token generated using a Unix timestamp as the seed can be predicted by searching a limited timestamp interval.

These experiments show that statistical randomness does not guarantee cryptographic security.

MT19937 should not be used for cryptographic keys, authentication tokens or other security-sensitive values.


## 13. Experimental Results

The following values are from the recorded terminal demonstrations. The exhaustive-search counts below reflect the corrected implementations, which continue after finding a match.

### 13.1 Part A: MT19937 Stream Cipher

The stream cipher was implemented using Python's `random` module, which uses MT19937. The plaintext contains a printable prefix followed by exactly 14 consecutive `A` characters.

**Generated Plaintext:**

```text
Q33IFZ8xuO+t94^iFAAAAAAAAAAAAAA
```

**Plaintext Length:** 31 bytes

**Original 16-bit Seed:** 46361

**Ciphertext (Hex):**

```text
dab5da2ec40e2c1d7e97c60b920751b56f8bd31c979128e98633b3157ec212
```

**Encryption/Decryption Verification:** True

### 13.2 Part A: Brute Force Seed Recovery

The corrected seed recovery implementation tests every possible 16-bit seed, even after identifying a valid candidate.

**Total Possible Seeds:** 65,536

**Total Seeds Tested:** 65,536

**Recovered Seed:** 46361

**Recovered Plaintext:**

```text
Q33IFZ8xuO+t94^iFAAAAAAAAAAAAAA
```

**Known Pattern Found:** True

**Full Plaintext Verification:** True

### 13.3 Part B: Predictable Token Generation

The token was generated by initializing MT19937 with the Unix timestamp and generating 16 consecutive 8-bit values.

**Unix Timestamp:** 1793861747

**Generated Token (Hex):**

```text
2f9793c53143d61681df76f209ce06f6
```

**Token Length:** 16 bytes (32 hexadecimal characters)

### 13.4 Part B: Timestamp Prediction Attack

The attacker searched the inclusive interval from `T - 300` through `T + 300`.

**Search Interval:** 1793861447 to 1793862047

**Total Possible Timestamps:** 601

**Total Timestamps Tested:** 601

**Recovered Timestamp:** 1793861747

**Regenerated Token:**

```text
2f9793c53143d61681df76f209ce06f6
```

**Token Verification:** True

### 13.5 Note on Reproducibility

The Python code cells in this notebook generate fresh random plaintext/seed values and a fresh timestamp when executed. Therefore, their outputs may differ from the recorded terminal example above. The important verification criteria are that Part A tests 65,536 seeds and recovers the seed/plaintext, and Part B tests 601 timestamps and regenerates the target token successfully.